# Dataset B: ESPN Official Conference Standings (2015-16 to 2025-26)

This notebook builds the **regular-season final standing (seed)** feature from `ListOfPotentialFeatures.md`, using the NBA's official order (with tiebreakers) instead of an approximation from win %:

| Feature | Column | Level |
|---|---|---|
| Regular-season final standing | `regular_season_standing` (1-15): the official order within the conference **at the end of the regular season**, before the play-in | team-season |
| Final playoff seed | `conference_seed` (1-15): the same order **after the play-in tournament**, so seeds 7-10 reflect the play-in results | team-season |

The two columns are identical except for the play-in teams (seeds 7-10) from 2020-21 on. Section 3D explains the difference.

*Run order: `01` → `02` → `999`. This notebook reads two raw files downloaded by notebook 01 (team ids and play-in games).*

- **Dataset name:** ESPN NBA standings
- **Link:** `https://site.api.espn.com/apis/v2/sports/basketball/nba/standings?season={end year}`. This is ESPN's public site API, the same family of endpoints used in `salary-data/03`.
- **Why ESPN and not NBA.com:** we first tried NBA.com's `leaguestandingsv3` endpoint (`stats.nba.com`), which has the same information. It does not respond to scripted requests from our network (every request hangs until it times out, with any headers we tried), so it can't be part of a notebook that downloads its own data. ESPN publishes the same official standings.
- **Saved as:** `data-files/00-raw/espn_standings_raw.json` (one API response per season, downloaded below if missing)
- **Also reads (from notebook 01):** `data-files/00-raw/llimllib_playoff_games_raw.csv` and `llimllib_nba_team_seasons_raw.csv`

## Important variables

| Variable | Units / values | Meaning |
|---|---|---|
| `season` | year | The year the season **ends** (2026 = 2025-26). We request each season by its end year. |
| `conference` | East / West | ESPN's "Eastern Conference" / "Western Conference". |
| `abbreviation` | text | ESPN's team abbreviation. A few differ from the NBA's (`GS`, `NY`, `SA`, `NO`, `UTAH`, `WSH`). |
| `wins`, `losses` | games | Regular-season record. |
| `playoffSeed` | 1-15 | ESPN's seed within the conference. |

## Concerns going in

1. **What does `playoffSeed` mean in play-in seasons?** Since 2020-21, the teams that finish 7th-10th play a play-in tournament for the 7 and 8 seeds. The seed could be the regular-season order or the order after the play-in. We test this in section 3D.
2. **Does ESPN apply the official tiebreakers?** If it does, teams with identical records should have distinct seeds, and a team should never be seeded above a team with a better record (except where the play-in changed the order).
3. **ESPN abbreviations differ from the NBA's** for six teams. We map them with `ESPN_TO_NBA_ABBREV` from `nba_cleaning_utils.py`.

## Setup

In [1]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import requests

# The shared helpers (paths, season range, ESPN abbreviation map) live next to the salary notebooks.
sys.path.insert(0, str(Path('..') / 'salary-data'))
from nba_cleaning_utils import RAW, PROCESSED, FIRST_SEASON, LAST_SEASON, ESPN_TO_NBA_ABBREV

## 1. Download (skipped if the raw file already exists)

In [2]:
RAW_STANDINGS = RAW / 'espn_standings_raw.json'
URL = 'https://site.api.espn.com/apis/v2/sports/basketball/nba/standings'

if not RAW_STANDINGS.exists():
    responses = []
    for season in range(FIRST_SEASON, LAST_SEASON + 1):
        r = requests.get(URL, params={'season': season}, timeout=60)
        r.raise_for_status()
        responses.append({'season': season, 'response': r.json()})
    RAW_STANDINGS.write_text(json.dumps(responses), encoding='utf-8')
    print('downloaded', len(responses), 'seasons')
else:
    print('raw file already present, skipping download')

downloaded 11 seasons


## 2. Load the data

Each response has one entry per conference, and each conference lists its 15 teams. Each team's numbers are a list of `{name, value}` pairs, which we turn into columns.

In [3]:
responses = json.loads(RAW_STANDINGS.read_text(encoding='utf-8'))

rows = []
for item in responses:
    for conf in item['response']['children']:
        for entry in conf['standings']['entries']:
            stats = {s['name']: s.get('value') for s in entry['stats']}
            rows.append({
                'season': item['season'],
                'conference': conf['name'],
                'espn_team_id': entry['team']['id'],
                'abbreviation': entry['team']['abbreviation'],
                'team_name': entry['team']['displayName'],
                **stats,
            })
standings_raw = pd.DataFrame(rows)

standings_raw[['season', 'conference', 'abbreviation', 'team_name', 'wins', 'losses', 'winPercent', 'playoffSeed']].head()

,season,conference,abbreviation,team_name,wins,losses,winPercent,playoffSeed
0,2016,Eastern Conference,CLE,Cleveland Cavaliers,57.0,25.0,0.695122,1.0
1,2016,Eastern Conference,MIA,Miami Heat,48.0,34.0,0.585366,3.0
2,2016,Eastern Conference,TOR,Toronto Raptors,56.0,26.0,0.682927,2.0
3,2016,Eastern Conference,DET,Detroit Pistons,44.0,38.0,0.536585,8.0
4,2016,Eastern Conference,IND,Indiana Pacers,45.0,37.0,0.548781,7.0


## 3A. Is the data tidy?

The observation is a team-season, so there should be one row per `season` + team.

In [4]:
print('columns:', standings_raw.shape[1])
print('duplicate (season, team) rows:', standings_raw.duplicated(['season', 'espn_team_id']).sum())
standings_raw[['season', 'wins', 'losses', 'winPercent', 'playoffSeed']].dtypes

columns: 28
duplicate (season, team) rows: 0


season           int64
wins           float64
losses         float64
winPercent     float64
playoffSeed    float64
dtype: object

**What we found:** After flattening, the table is tidy: one row per team-season, one column per stat, with no duplicate keys. ESPN returns about 25 stats per team, such as home/road records and points for and against. We only need the record and `playoffSeed`; the rest are dropped in section 4. The numbers load as floats and are converted to integers there.

## 3B. Size of the data

In [5]:
print('rows, columns:', standings_raw.shape)
standings_raw.groupby(['season', 'conference']).size().unstack()

rows, columns: (330, 28)


conference,Eastern Conference,Western Conference
season,,
2016,15,15
2017,15,15
2018,15,15
2019,15,15
2020,15,15
2021,15,15
2022,15,15
2023,15,15
2024,15,15


**330 team-seasons**: 11 seasons, with 15 teams in each conference every season, matching our 2015-16 to 2025-26 window. Only these seasons were requested, so nothing needs to be dropped.

## 3C. Missing data

In [6]:
standings_raw[['season', 'conference', 'abbreviation', 'wins', 'losses', 'playoffSeed']].isna().sum()

season          0
conference      0
abbreviation    0
wins            0
losses          0
playoffSeed     0
dtype: int64

**What we found:** No blanks in any of the columns we use.

## 3D. Outliers and suspicious entries

**(a) Is each conference's seed list a clean 1-15?** Repeated or skipped seeds would mean ESPN didn't break ties.

In [7]:
st = standings_raw.copy()
st['seed'] = st['playoffSeed'].astype(int)
st['win_pct'] = st['wins'] / (st['wins'] + st['losses'])
clean_list = st.groupby(['season', 'conference'])['seed'].apply(lambda s: sorted(s) == list(range(1, 16)))
print('conference-seasons with seeds exactly 1-15:', clean_list.sum(), 'of', len(clean_list))

conference-seasons with seeds exactly 1-15: 22 of 22


**(b) Does the seed order follow the records?** We list every pair where a team is seeded **above** a team with a strictly **better** win %. With only regular-season standings and tiebreakers, this should never happen, because tiebreakers only order teams with identical records.

In [8]:
def order_breaks(df, col):
    # Pairs (higher-ranked team, lower-ranked team) where the higher-ranked team has the worse win %.
    out = []
    for (season, conf), g in df.groupby(['season', 'conference']):
        g = g.sort_values(col)
        for i in range(len(g)):
            better_below = g.iloc[i + 1:][g['win_pct'].iloc[i + 1:] > g['win_pct'].iloc[i] + 1e-12]
            for _, b in better_below.iterrows():
                a = g.iloc[i]
                out.append({'season': season, 'conference': conf,
                            'above': a['abbreviation'], 'above_rank': a[col], 'above_W-L': f"{a['wins']:.0f}-{a['losses']:.0f}",
                            'below': b['abbreviation'], 'below_rank': b[col], 'below_W-L': f"{b['wins']:.0f}-{b['losses']:.0f}"})
    return pd.DataFrame(out)

breaks = order_breaks(st, 'seed')
print('pairs seeded against the records:', len(breaks))
breaks

pairs seeded against the records: 10


,season,conference,above,above_rank,above_W-L,below,below_rank,below_W-L
0,2021,Western Conference,MEM,8,38-34,GS,9,39-33
1,2022,Eastern Conference,ATL,8,43-39,CLE,9,44-38
2,2022,Western Conference,NO,8,36-46,LAC,9,42-40
3,2023,Eastern Conference,ATL,7,41-41,MIA,8,44-38
4,2023,Eastern Conference,CHI,9,40-42,TOR,10,41-41
5,2023,Western Conference,OKC,9,40-42,NO,10,42-40
6,2024,Western Conference,LAL,7,47-35,NO,8,49-33
7,2025,Eastern Conference,MIA,8,37-45,ATL,9,40-42
8,2025,Eastern Conference,MIA,8,37-45,CHI,10,39-43
9,2026,Western Conference,POR,7,42-40,PHX,8,45-37


In [9]:
(st.query("season == 2021 and conference == 'Western Conference' and 5 <= seed <= 11")
   .sort_values('seed')[['abbreviation', 'wins', 'losses', 'seed']])

,abbreviation,wins,losses,seed
165,DAL,42.0,30.0,5
169,POR,42.0,30.0,6
168,LAL,42.0,30.0,7
167,MEM,38.0,34.0,8
173,GS,39.0,33.0,9
172,SA,33.0,39.0,10
177,NO,31.0,41.0,11


**What we found:**

- **Every conference-season has seeds exactly 1-15**, so ESPN breaks ties between identical records.
- **There are 10 pairs where a team is seeded above a team with a better record, and all of them are among seeds 7-10 in play-in seasons (2020-21 and later).** Outside seeds 7-10, ESPN's order always follows the records, as the official standings must.
- **So in play-in seasons, `playoffSeed` is the seed after the play-in, not the regular-season standing.** For example, in 2020-21 the West's regular-season 7th-10th place teams were LAL (42-30), GSW (39-33), MEM (38-34), and SAS (33-39). LAL beat GSW for the 7 seed, then MEM beat GSW for the 8 seed. ESPN lists MEM as the 8 seed and GSW as the 9 seed, even though GSW had the better record. The play-in losers are also re-ordered: the loser of the 8-seed game is 9, and the loser of the 9-vs-10 game is 10.

Both versions are useful, since one is how the team actually finished the regular season and the other is the seed it carried into the playoffs. So we keep ESPN's value as `conference_seed` and rebuild the regular-season order as `regular_season_standing` (section 4B).

No other suspicious entries were found.

## 4. Clean the data

### 4A. Keys and columns

Map ESPN's abbreviations to the NBA's, shorten the conference names to match notebook 01 (`East` / `West`), and keep only the columns we need. We keep ESPN's wins and losses as `espn_wins` / `espn_losses` so notebook 999 can check them against Basketball-Reference.

In [10]:
standings = (st.assign(team=st['abbreviation'].replace(ESPN_TO_NBA_ABBREV),
                       conference=st['conference'].map({'Eastern Conference': 'East', 'Western Conference': 'West'}),
                       espn_wins=st['wins'].astype(int),
                       espn_losses=st['losses'].astype(int))
               .rename(columns={'seed': 'conference_seed'})
               [['season', 'team', 'espn_team_id', 'conference', 'espn_wins', 'espn_losses', 'win_pct', 'conference_seed']])

print('teams:', standings['team'].nunique(), sorted(standings['team'].unique()))
assert standings['conference'].notna().all() and not standings.duplicated(['season', 'team']).any()

teams: 30 ['ATL', 'BKN', 'BOS', 'CHA', 'CHI', 'CLE', 'DAL', 'DEN', 'DET', 'GSW', 'HOU', 'IND', 'LAC', 'LAL', 'MEM', 'MIA', 'MIL', 'MIN', 'NOP', 'NYK', 'OKC', 'ORL', 'PHI', 'PHX', 'POR', 'SAC', 'SAS', 'TOR', 'UTA', 'WAS']


### 4B. Rebuild the regular-season standing for the play-in teams

The play-in bracket shows the regular-season order of the four teams it involves. In the first round, **7th place hosts 8th place, and 9th place hosts 10th place**. We take the first-round play-in games from notebook 01's raw `games` file:

- The first-round game that includes ESPN's 7 seed is the 7-vs-8 game (its winner is the 7 seed). Its home team finished **7th** and its away team **8th**.
- The other first-round game in that conference is the 9-vs-10 game. Its home team finished **9th** and its away team **10th**.
- **2019-20 is a special case.** The bubble had a single West play-in series between the 8th- and 9th-place teams (POR vs MEM). POR, the 8th-place team, won, so the order didn't change. Here the two teams keep their own seeds.

Every other team's regular-season standing equals its `conference_seed`.

In [11]:
ids = {'team_id': str, 'home_team_id': str, 'away_team_id': str, 'game_id': str}
games = pd.read_csv(RAW / 'llimllib_playoff_games_raw.csv', dtype=ids)
team_ids = pd.read_csv(RAW / 'llimllib_nba_team_seasons_raw.csv', dtype=ids)[['season', 'team_id', 'nba_abbrev']]

play_in = games.query("game_type == 'play_in' and playoff_round == 1 and @FIRST_SEASON <= season <= @LAST_SEASON")
for side in ['home', 'away']:
    play_in = play_in.merge(team_ids.rename(columns={'team_id': f'{side}_team_id', 'nba_abbrev': side}),
                            on=['season', f'{side}_team_id'], how='left', validate='many_to_one')
    play_in = play_in.merge(standings[['season', 'team', 'conference', 'conference_seed']]
                            .rename(columns={'team': side, 'conference': f'{side}_conf', 'conference_seed': f'{side}_seed'}),
                            on=['season', side], how='left', validate='many_to_one')
assert (play_in['home_conf'] == play_in['away_conf']).all()
play_in = play_in.rename(columns={'home_conf': 'conference'})

slots = []
for (season, conf), g in play_in.groupby(['season', 'conference']):
    if len(g) == 1:                       # 2019-20 bubble: one 8-vs-9 series, the order didn't change
        game = g.iloc[0]
        seeds = sorted([game['home_seed'], game['away_seed']])
        slots += [(season, game['home'], seeds[0]), (season, game['away'], seeds[1])]
        continue
    assert len(g) == 2, (season, conf)
    is_7v8 = (g['home_seed'] == 7) | (g['away_seed'] == 7)
    assert is_7v8.sum() == 1, (season, conf)
    for game, (high, low) in [(g[is_7v8].iloc[0], (7, 8)), (g[~is_7v8].iloc[0], (9, 10))]:
        slots += [(season, game['home'], high), (season, game['away'], low)]
slots = pd.DataFrame(slots, columns=['season', 'team', 'play_in_slot'])

standings = standings.merge(slots, on=['season', 'team'], how='left', validate='one_to_one')
standings['regular_season_standing'] = standings['play_in_slot'].fillna(standings['conference_seed']).astype(int)
standings = standings.drop(columns='play_in_slot')

play_in[['season', 'conference', 'home', 'home_seed', 'away', 'away_seed']].sort_values(['season', 'conference'])

,season,conference,home,home_seed,away,away_seed
0,2020,West,POR,8,MEM,9
1,2021,East,BOS,7,WAS,8
2,2021,East,IND,9,CHA,10
3,2021,West,LAL,7,GSW,9
4,2021,West,MEM,8,SAS,10
5,2022,East,BKN,7,CLE,9
6,2022,East,ATL,8,CHA,10
7,2022,West,MIN,7,LAC,9
8,2022,West,NOP,8,SAS,10
9,2023,East,MIA,8,ATL,7


### 4C. Check the rebuilt standings

The regular-season standing should be a clean 1-15 in every conference-season and **never** put a team above a team with a better record. The play-in teams should also be exactly the teams ESPN seeds 7-10.

In [12]:
check = standings.rename(columns={'team': 'abbreviation', 'espn_wins': 'wins', 'espn_losses': 'losses'})
print('conference-seasons with standings exactly 1-15:',
      standings.groupby(['season', 'conference'])['regular_season_standing'].apply(lambda s: sorted(s) == list(range(1, 16))).sum(), 'of 22')
print('pairs ranked against the records:', len(order_breaks(check, 'regular_season_standing')))
moved = standings[standings['regular_season_standing'] != standings['conference_seed']]
print('team-seasons where the play-in changed the order:', len(moved))
print('...all within seeds 7-10:', moved['conference_seed'].between(7, 10).all() and moved['regular_season_standing'].between(7, 10).all())

(standings.query("season == 2024 and conference == 'West' and 5 <= conference_seed <= 11")
          .sort_values('regular_season_standing')
          [['team', 'espn_wins', 'espn_losses', 'regular_season_standing', 'conference_seed']])

conference-seasons with standings exactly 1-15: 22 of 22


pairs ranked against the records: 0
team-seasons where the play-in changed the order: 19
...all within seeds 7-10: True


,team,espn_wins,espn_losses,regular_season_standing,conference_seed
256,DAL,50,32,5,5
260,PHX,49,33,6,6
259,NOP,49,33,7,8
258,LAL,47,35,8,7
263,SAC,46,36,9,9
264,GSW,46,36,10,10
269,HOU,41,41,11,11


**What we found:** The rebuilt `regular_season_standing` is a clean 1-15 in all 22 conference-seasons and **never ranks a team above a team with a better record**. In other words, it orders by record and separates identical records with the official tiebreakers. Only play-in teams moved between the two columns. The 2023-24 West shows the difference: NOP (49-33) finished **7th** and LAL (47-35) **8th**. LAL won their play-in game and entered the playoffs as the **7 seed**, while NOP lost and then beat SAC to enter as the 8 seed.

## 5. Save

In [13]:
standings_features = (standings[['season', 'team', 'espn_team_id', 'conference', 'espn_wins', 'espn_losses',
                                 'regular_season_standing', 'conference_seed']]
                      .sort_values(['season', 'team']).reset_index(drop=True))
out = PROCESSED / 'espn_standings_features.csv'
standings_features.to_csv(out, index=False)
print('saved', standings_features.shape, 'to', out)
standings_features.head()

saved (330, 8) to ..\..\data-files\02-processed\espn_standings_features.csv


,season,team,espn_team_id,conference,espn_wins,espn_losses,regular_season_standing,conference_seed
0,2016,ATL,1,East,48,34,4,4
1,2016,BKN,17,East,21,61,14,14
2,2016,BOS,2,East,48,34,5,5
3,2016,CHA,30,East,48,34,6,6
4,2016,CHI,4,East,42,40,9,9


**Output:** `data-files/02-processed/espn_standings_features.csv`, **one row per team + season** (330 rows).

| Column | Meaning |
|---|---|
| `season`, `team` | join keys (end year, NBA abbreviation) |
| `espn_team_id`, `conference` | ESPN team id; East / West |
| `espn_wins`, `espn_losses` | ESPN's regular-season record, kept so notebook 999 can check it against Basketball-Reference |
| `regular_season_standing` | official order at the end of the regular season (1-15) |
| `conference_seed` | official seed after the play-in (1-15); differs from `regular_season_standing` only for some play-in teams from 2020-21 on |